
# Descriptive statistics tables for treated and control municipalities

This notebook loads the **fixed analysis-ready municipality-quarter panel** from `ROOT/out`, constructs a pooled **treated vs control** grouping, and creates **two descriptive-statistics tables**:

1. **Full-sample table**  
   Uses all municipalities in the analysis-ready panel, but each statistic is computed on the valid sample for that statistic.

2. **Rent-sample table**  
   Restricts both treated and control groups to municipalities that **have rent data**.

The rows are:

- Average Rent Price Pre
- Average Rent Price Post
- Yearly Rent Growth (2011-2025)
- Yearly Rent Growth (2022-2024)
- Average Rent Growth Post
- Tenancy Agreements per 10k Pre
- Tenancy Agreements per 10k Post
- Rent Price/Taxable Base
- Yearly Taxable Base
- Population
- Natives
- Foreigners
- Net Inflows
- Net Outflows
- Number of Municipalities
- Municipalities with Data on Rents

**Important choice:** for this pooled descriptive table, `Pre`/`Post` are defined relative to the **first implementation quarter**, with:

- `Pre`: before **2024-Q2**
- `Post`: from **2024-Q2** onward

That is a descriptive convention for the table, not the staggered-treatment estimator itself.


In [1]:

from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [2]:

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate
    return start

ROOT = find_project_root()
OUT_DIR = ROOT / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

candidate_paths = [
    OUT_DIR / "analysis_ready_panel_fixed.pkl",
    OUT_DIR / "analysis_ready_panel_fixed.jsonl",
    OUT_DIR / "analysis_ready_panel.pkl",
    OUT_DIR / "analysis_ready_panel.jsonl",
]

panel_path = next((p for p in candidate_paths if p.exists()), None)
if panel_path is None:
    raise FileNotFoundError(
        "Could not find analysis_ready_panel_fixed.* or analysis_ready_panel.* in ROOT/out"
    )

print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)
print("Loaded path:", panel_path)


ROOT: /home/venatio/documents/university/public-sector-2026
OUT_DIR: /home/venatio/documents/university/public-sector-2026/out
Loaded path: /home/venatio/documents/university/public-sector-2026/out/analysis_ready_panel.pkl


In [3]:

if panel_path.suffix == ".pkl":
    panel = pd.read_pickle(panel_path)
elif panel_path.suffix == ".jsonl":
    panel = pd.read_json(panel_path, orient="records", lines=True)
else:
    panel = pd.read_csv(panel_path)

panel.columns = [str(c).strip() for c in panel.columns]

print("Shape:", panel.shape)
print("Columns:", panel.columns.tolist())
display(panel.head())


Shape: (56908, 61)
Columns: ['Code', 'Municipality', 'municipality_norm', 'Year', 'Quarter', 'quarter_num', 'time_index', 'Prices', 'Contracts', 'price_observed_raw', 'contracts_observed_raw', 'price_imputed', 'contracts_imputed', 'has_any_observed_price', 'has_any_observed_contracts', 'exclude_from_price_analysis', 'exclude_from_contract_analysis', 'has_valid_price_for_analysis', 'has_valid_contracts_for_analysis', 'Treated 1', 'Treated 2', 'Control', 'ever_treated', 'cohort', 'first_treat_period', 'first_treat_index', 'treated', 'event_time', 'pop_total', 'pop_males', 'pop_females', 'pop_native_total', 'pop_native_males', 'pop_native_females', 'pop_foreign_total', 'pop_foreign_males', 'pop_foreign_females', 'mig_total', 'mig_total_males', 'mig_total_females', 'mig_inflows', 'mig_inflows_males', 'mig_inflows_females', 'mig_outflows', 'mig_outflows_males', 'mig_outflows_females', 'income_tax_base', 'income_tax_liability', 'log_prices', 'price_growth_qoq_pct', 'price_growth_yoy_pct', 'l

,Code,Municipality,municipality_norm,Year,Quarter,quarter_num,time_index,Prices,Contracts,price_observed_raw,contracts_observed_raw,price_imputed,contracts_imputed,has_any_observed_price,has_any_observed_contracts,exclude_from_price_analysis,exclude_from_contract_analysis,has_valid_price_for_analysis,has_valid_contracts_for_analysis,Treated 1,Treated 2,Control,ever_treated,cohort,first_treat_period,first_treat_index,treated,event_time,pop_total,pop_males,pop_females,pop_native_total,pop_native_males,pop_native_females,pop_foreign_total,pop_foreign_males,pop_foreign_females,mig_total,mig_total_males,mig_total_females,mig_inflows,mig_inflows_males,mig_inflows_females,mig_outflows,mig_outflows_males,mig_outflows_females,income_tax_base,income_tax_liability,log_prices,price_growth_qoq_pct,price_growth_yoy_pct,log_price_growth_qoq,log_price_growth_yoy,contracts_per_10k,contracts_per_10k_qoq_pct,contracts_per_10k_yoy_pct,rent_to_income_proxy,annualized_rent_to_income_proxy,rent_to_tax_liability_proxy,outcome_price_growth,outcome_market_activity
0,00005,NaN,,2015,Q1,1,8061,NaN,NaN,0,0,0,0,0,0,1,1,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00005,NaN,,2015,Q2,2,8062,NaN,NaN,0,0,0,0,0,0,1,1,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00005,NaN,,2015,Q3,3,8063,NaN,NaN,0,0,0,0,0,0,1,1,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00005,NaN,,2015,Q4,4,8064,NaN,NaN,0,0,0,0,0,0,1,1,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,02050,NaN,,2011,Q1,1,8045,NaN,NaN,0,0,0,0,0,0,1,1,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:

required_cols = [
    "Code", "Municipality", "Year", "Quarter",
    "Prices", "Contracts",
    "pop_total", "pop_native_total", "pop_foreign_total",
    "mig_inflows", "mig_outflows",
    "income_tax_base", "rent_to_income_proxy",
    "price_growth_yoy_pct",
]

missing_required = [c for c in required_cols if c not in panel.columns]
if missing_required:
    raise ValueError(f"Missing required columns in analysis-ready panel: {missing_required}")

if "ever_treated" in panel.columns:
    panel["Group"] = np.where(panel["ever_treated"].fillna(0).astype(int) == 1, "Treated", "Control")
elif {"Treated 1", "Treated 2"}.issubset(panel.columns):
    ever_treated = (
        panel["Treated 1"].fillna(0).astype(int) == 1
    ) | (
        panel["Treated 2"].fillna(0).astype(int) == 1
    )
    panel["Group"] = np.where(ever_treated, "Treated", "Control")
else:
    raise ValueError("Could not identify treated vs control groups from the loaded panel.")

if "has_any_observed_price" not in panel.columns:
    panel["has_any_observed_price"] = (
        panel.groupby("Code")["Prices"].transform(lambda s: int(s.notna().any()))
    )

if "exclude_from_price_analysis" not in panel.columns:
    panel["exclude_from_price_analysis"] = (
        panel["has_any_observed_price"].fillna(0).astype(int) == 0
    ).astype(int)

quarter_num_map = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4}
panel["quarter_num"] = panel["Quarter"].map(quarter_num_map)
bad_quarters = sorted(set(panel.loc[panel["quarter_num"].isna(), "Quarter"].dropna()))
if bad_quarters:
    raise ValueError(f"Unexpected quarter labels found: {bad_quarters}")

panel["Year"] = pd.to_numeric(panel["Year"], errors="coerce").astype("Int64")
panel["time_index"] = panel["Year"].astype(int) * 4 + panel["quarter_num"].astype(int)

POST_START_YEAR = 2024
POST_START_QUARTER = "Q2"
POST_START_INDEX = POST_START_YEAR * 4 + quarter_num_map[POST_START_QUARTER]

panel["is_pre"] = panel["time_index"] < POST_START_INDEX
panel["is_post"] = panel["time_index"] >= POST_START_INDEX

numeric_cols = [
    "Prices", "Contracts",
    "contracts_per_10k",
    "pop_total", "pop_native_total", "pop_foreign_total",
    "mig_inflows", "mig_outflows",
    "income_tax_base", "rent_to_income_proxy",
    "price_growth_yoy_pct",
]
for c in numeric_cols:
    if c in panel.columns:
        panel[c] = pd.to_numeric(panel[c], errors="coerce")

if "contracts_per_10k" not in panel.columns:
    panel["contracts_per_10k"] = 10000 * panel["Contracts"] / panel["pop_total"]

rent_metric_cols = ["Prices", "price_growth_yoy_pct", "rent_to_income_proxy"]
for c in rent_metric_cols:
    if c in panel.columns:
        panel.loc[panel["exclude_from_price_analysis"] == 1, c] = np.nan

panel = panel.sort_values(["Code", "Year", "quarter_num"]).reset_index(drop=True)

display(panel[[
    "Code", "Municipality", "Year", "Quarter", "Group",
    "is_pre", "is_post", "has_any_observed_price", "exclude_from_price_analysis"
]].head(12))


,Code,Municipality,Year,Quarter,Group,is_pre,is_post,has_any_observed_price,exclude_from_price_analysis
0,00005,NaN,2015,Q1,Control,True,False,0,1
1,00005,NaN,2015,Q2,Control,True,False,0,1
2,00005,NaN,2015,Q3,Control,True,False,0,1
3,00005,NaN,2015,Q4,Control,True,False,0,1
4,02050,NaN,2011,Q1,Control,True,False,0,1
5,02050,NaN,2011,Q2,Control,True,False,0,1
6,02050,NaN,2011,Q3,Control,True,False,0,1
7,02050,NaN,2011,Q4,Control,True,False,0,1
8,02050,NaN,2012,Q1,Control,True,False,0,1
9,02050,NaN,2012,Q2,Control,True,False,0,1



## Table construction strategy

For continuous variables, each table uses **equal municipality weighting**:

1. Compute each municipality's mean over the relevant sample window.
2. Average those municipality-level means within `Treated` and `Control`.

This prevents municipalities with more valid rows from receiving extra weight.

For counts:
- `Number of Municipalities` is the number of unique municipalities in each group.
- `Municipalities with Data on Rents` is the number of unique municipalities with at least one observed rent-price series.

The second table restricts the sample to municipalities with `has_any_observed_price == 1`.


In [5]:

def municipality_mean_by_group(df: pd.DataFrame, value_col: str, mask=None) -> pd.Series:
    tmp = df.copy()
    if mask is not None:
        tmp = tmp.loc[mask].copy()

    tmp = tmp[["Code", "Group", value_col]].copy()
    tmp[value_col] = pd.to_numeric(tmp[value_col], errors="coerce")
    tmp = tmp.dropna(subset=[value_col])

    if tmp.empty:
        return pd.Series({"Treated": np.nan, "Control": np.nan}, dtype=float)

    muni_means = tmp.groupby(["Code", "Group"], as_index=False)[value_col].mean()
    group_means = muni_means.groupby("Group")[value_col].mean()

    return pd.Series({
        "Treated": group_means.get("Treated", np.nan),
        "Control": group_means.get("Control", np.nan),
    })


def municipality_count_by_group(df: pd.DataFrame, mask=None) -> pd.Series:
    tmp = df.copy()
    if mask is not None:
        tmp = tmp.loc[mask].copy()

    counts = tmp.groupby("Group")["Code"].nunique()
    return pd.Series({
        "Treated": counts.get("Treated", 0),
        "Control": counts.get("Control", 0),
    })


def municipality_count_with_rent_data(df: pd.DataFrame, mask=None) -> pd.Series:
    tmp = df.copy()
    if mask is not None:
        tmp = tmp.loc[mask].copy()

    if "has_any_observed_price" in tmp.columns:
        tmp = tmp[tmp["has_any_observed_price"].fillna(0).astype(int) == 1].copy()
    else:
        tmp = tmp[tmp["Prices"].notna()].copy()

    counts = tmp.groupby("Group")["Code"].nunique()
    return pd.Series({
        "Treated": counts.get("Treated", 0),
        "Control": counts.get("Control", 0),
    })


def build_descriptive_table(df: pd.DataFrame) -> pd.DataFrame:
    stats_rows = {
        "Average Rent Price Pre": municipality_mean_by_group(df, "Prices", df["is_pre"]),
        "Average Rent Price Post": municipality_mean_by_group(df, "Prices", df["is_post"]),
        "Yearly Rent Growth (2011-2025)": municipality_mean_by_group(
            df,
            "price_growth_yoy_pct",
            df["Year"].between(2011, 2025),
        ),
        "Yearly Rent Growth (2022-2024)": municipality_mean_by_group(
            df,
            "price_growth_yoy_pct",
            df["Year"].between(2022, 2024),
        ),
        "Average Rent Growth Post": municipality_mean_by_group(
            df,
            "price_growth_yoy_pct",
            df["is_post"],
        ),
        "Tenancy Agreements per 10k Pre": municipality_mean_by_group(
            df,
            "contracts_per_10k",
            df["is_pre"],
        ),
        "Tenancy Agreements per 10k Post": municipality_mean_by_group(
            df,
            "contracts_per_10k",
            df["is_post"],
        ),
        "Rent Price/Taxable Base": municipality_mean_by_group(df, "rent_to_income_proxy"),
        "Yearly Taxable Base": municipality_mean_by_group(df, "income_tax_base"),
        "Population": municipality_mean_by_group(df, "pop_total"),
        "Natives": municipality_mean_by_group(df, "pop_native_total"),
        "Foreigners": municipality_mean_by_group(df, "pop_foreign_total"),
        "Net Inflows": municipality_mean_by_group(df, "mig_inflows"),
        "Net Outflows": municipality_mean_by_group(df, "mig_outflows"),
        "Number of Municipalities": municipality_count_by_group(df),
        "Municipalities with Data on Rents": municipality_count_with_rent_data(df),
    }

    desc = pd.DataFrame(stats_rows).T.reset_index().rename(columns={"index": "Statistic"})
    return desc[["Statistic", "Treated", "Control"]]


count_rows = {
    "Number of Municipalities",
    "Municipalities with Data on Rents",
}

def format_for_display(desc_table: pd.DataFrame) -> pd.DataFrame:
    out = desc_table.copy()
    for col in ["Treated", "Control"]:
        out[col] = out.apply(
            lambda r: int(round(r[col])) if r["Statistic"] in count_rows and pd.notna(r[col]) else r[col],
            axis=1,
        )
    return out


In [6]:

desc_table = build_descriptive_table(panel)
desc_table_display = format_for_display(desc_table)

print("Full-sample descriptive table")
display(desc_table_display)


Full-sample descriptive table


,Statistic,Treated,Control
0,Average Rent Price Pre,555.9226,459.2492
1,Average Rent Price Post,718.6952,556.2429
2,Yearly Rent Growth (2011-2025),3.1745,2.5673
3,Yearly Rent Growth (2022-2024),6.0793,4.2416
4,Average Rent Growth Post,2.9524,3.6889
5,Tenancy Agreements per 10k Pre,39.2481,21.2272
6,Tenancy Agreements per 10k Post,30.4929,21.1986
7,Rent Price/Taxable Base,0.0248,0.0226
8,Yearly Taxable Base,"22,847.3680","19,749.9643"
9,Population,"25,624.9658","1,062.9853"


In [ ]:

panel_rent_sample = panel.loc[
    panel["has_any_observed_price"].fillna(0).astype(int) == 1
].copy()

desc_table_rent_sample = build_descriptive_table(panel_rent_sample)
desc_table_rent_sample_display = format_for_display(desc_table_rent_sample)

print("Rent-sample descriptive table")
display(desc_table_rent_sample_display)

Rent-sample descriptive table


,Statistic,Treated,Control
0,Average Rent Price Pre,555.9226,459.2492
1,Average Rent Price Post,718.6952,556.2429
2,Yearly Rent Growth (2011-2025),3.1745,2.5673
3,Yearly Rent Growth (2022-2024),6.0793,4.2416
4,Average Rent Growth Post,2.9524,3.6889
5,Tenancy Agreements per 10k Pre,39.2481,29.0354
6,Tenancy Agreements per 10k Post,30.4929,28.6870
7,Rent Price/Taxable Base,0.0248,0.0226
8,Yearly Taxable Base,"22,847.3680","20,892.1940"
9,Population,"25,624.9658","2,170.0137"


In [8]:

OUT_CSV = OUT_DIR / "descriptive_stats_treated_control_fixed.csv"
OUT_XLSX = OUT_DIR / "descriptive_stats_treated_control_fixed.xlsx"
OUT_LATEX = OUT_DIR / "descriptive_stats_treated_control_fixed.tex"

OUT_CSV_RENT = OUT_DIR / "descriptive_stats_treated_control_rent_sample.csv"
OUT_XLSX_RENT = OUT_DIR / "descriptive_stats_treated_control_rent_sample.xlsx"
OUT_LATEX_RENT = OUT_DIR / "descriptive_stats_treated_control_rent_sample.tex"

desc_table_display.to_csv(OUT_CSV, index=False)
desc_table_display.to_excel(OUT_XLSX, index=False)
OUT_LATEX.write_text(
    desc_table_display.to_latex(index=False, escape=False, float_format="%.4f"),
    encoding="utf-8",
)

desc_table_rent_sample_display.to_csv(OUT_CSV_RENT, index=False)
desc_table_rent_sample_display.to_excel(OUT_XLSX_RENT, index=False)
OUT_LATEX_RENT.write_text(
    desc_table_rent_sample_display.to_latex(index=False, escape=False, float_format="%.4f"),
    encoding="utf-8",
)

print("Saved:")
print(" -", OUT_CSV)
print(" -", OUT_XLSX)
print(" -", OUT_LATEX)
print(" -", OUT_CSV_RENT)
print(" -", OUT_XLSX_RENT)
print(" -", OUT_LATEX_RENT)

Saved:
 - /home/venatio/documents/university/public-sector-2026/out/descriptive_stats_treated_control_fixed.csv
 - /home/venatio/documents/university/public-sector-2026/out/descriptive_stats_treated_control_fixed.xlsx
 - /home/venatio/documents/university/public-sector-2026/out/descriptive_stats_treated_control_fixed.tex
 - /home/venatio/documents/university/public-sector-2026/out/descriptive_stats_treated_control_rent_sample.csv
 - /home/venatio/documents/university/public-sector-2026/out/descriptive_stats_treated_control_rent_sample.xlsx
 - /home/venatio/documents/university/public-sector-2026/out/descriptive_stats_treated_control_rent_sample.tex


In [9]:

print("Unique municipalities by group (full panel):")
display(
    panel.groupby("Group")["Code"].nunique().rename("n_municipalities").reset_index()
)

print("Unique municipalities by group (rent sample only):")
display(
    panel_rent_sample.groupby("Group")["Code"].nunique().rename("n_municipalities").reset_index()
)

print("Municipalities with rent data by group (full panel):")
display(
    panel.groupby(["Group", "Code"], as_index=False)["has_any_observed_price"].max()
         .groupby("Group", as_index=False)["has_any_observed_price"].sum()
         .rename(columns={"has_any_observed_price": "n_with_rent_data"})
)

print("Share of non-missing Prices by group within the rent sample:")
display(
    panel_rent_sample.assign(has_price=panel_rent_sample["Prices"].notna().astype(int))
                     .groupby("Group", as_index=False)["has_price"]
                     .mean()
)

Unique municipalities by group (full panel):


,Group,n_municipalities
0,Control,679
1,Treated,271


Unique municipalities by group (rent sample only):


,Group,n_municipalities
0,Control,257
1,Treated,271


Municipalities with rent data by group (full panel):


,Group,n_with_rent_data
0,Control,257
1,Treated,271


Share of non-missing Prices by group within the rent sample:


,Group,has_price
0,Control,1.0000
1,Treated,1.0000
